# Análise Estocástica: Simulação de Caminhadas Aleatórias e o Ajuste da Gaussiana

Este notebook interativo apresenta a simulação de caminhadas aleatórias unidimensionais discretas. O objetivo é demonstrar visualmente a evolução temporal dessas caminhadas e comprovar o **Teorema Central do Limite (TLC)**, mostrando como a distribuição empírica das posições finais converge para uma distribuição contínua Gaussiana (Normal) quando $n \gg 1$.

---

## 1. Fundamentação Teórica

Em uma caminhada aleatória unidimensional discreta, um caminhante parte da origem ($x = 0$) e, a cada intervalo de tempo, dá um passo elementar $X_i$ que pode ser:
* $+1$ (direita) com probabilidade $p$
* $-1$ (esquerda) com probabilidade $1-p$

A posição final após $n$ passos é a soma acumulada de cada passo individual:

$$S_n = \sum_{i=1}^{n} X_i$$

### O Problema da Paridade e o Espaço Discreto ($\Delta x = 2$)
Se definirmos $n_+$ como o número de passos para a direita e $n_-$ como o número de passos para a esquerda, sabemos que $n_+ + n_- = n$. Como a posição final é $S_n = n_+ - n_-$, podemos reescrever a posição como:

$$S_n = 2n_+ - n$$

Isso nos impõe duas propriedades matemáticas cruciais que afetam a nossa análise:
1. **Restrição de Paridade:** A paridade da posição final $S_n$ será sempre idêntica à paridade do número total de passos $n$. Se $n = 100$ (par), as posições finais só podem ser números pares (ex: $-4, 0, 2, 8$). É impossível terminar em uma posição ímpar.
2. **Espaçamento da Rede:** Como $n_+$ varia de 1 em 1 unidade inteira, a posição final $S_n$ varia de **2 em 2 unidades**. Portanto, a distância entre duas posições permitidas consecutivas na rede é **$\Delta x = 2$**.

### Densidade de Probabilidade vs. Massa de Probabilidade
A aproximação contínua da Gaussiana calcula a **Função Densidade de Probabilidade (PDF)**, cuja área total integrada sob a curva é igual a 1:

$$\int_{-\infty}^{\infty} f(x) dx = 1$$

Por outro lado, a simulação e a distribuição exata (Binomial) lidam com **Massa de Probabilidade (PMF)**, onde a soma das probabilidades de pontos isolados deve ser igual a 1.

Para ajustar a curva teórica contínua $f(x)$ aos pontos discretos, precisamos aproximar a probabilidade integrando a densidade em um intervalo correspondente à largura do bloco de cada ponto válido. Como o nosso espaço discreto caminha com $\Delta x = 2$, a probabilidade de a caminhada terminar exatamente na posição válida $x$ é dada pela área do retângulo de largura 2 centrado em $x$:

$$P(X = x) \approx f(x) \cdot \Delta x = f(x) \cdot 2$$

Por isso, no código, multiplicamos a curva teórica da Gaussiana por 2 para que ela se sobreponha perfeitamente às probabilidades discretas.

### Parâmetros Estatísticos
* **Média Esperada ($\mu$):** $\mu = n(2p - 1)$
* **Variância ($\sigma^2$):** $\sigma^2 = 4np(1-p)$
* **Desvio Padrão ($\sigma$):** $\sigma = \sqrt{4np(1-p)}$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, binom

# --- Parâmetros ---
n = 1000     # Número de passos (n >> 1)
m = 10000    # Número de realizações
p = 0.7      # Probabilidade de passo à direita

# =====================================================================
# 1. Evolução de UMA caminhada aleatória
# =====================================================================
passos_1 = np.where(np.random.rand(n) < p, 1, -1)
caminhada_1 = np.concatenate(([0], np.cumsum(passos_1)))

plt.figure(figsize=(10, 4))
plt.plot(caminhada_1, color='blue', linewidth=1)
plt.title(f'Evolução de 1 Caminhada Aleatória ({n} passos)')
plt.xlabel('Número de Passos')
plt.ylabel('Posição')
plt.grid(True, alpha=0.3)
plt.show()

# =====================================================================
# 2. Evolução de M caminhadas aleatórias
# =====================================================================
passos_m = np.where(np.random.rand(m, n) < p, 1, -1)
caminhadas_m = np.concatenate((np.zeros((m, 1)), np.cumsum(passos_m, axis=1)), axis=1)

plt.figure(figsize=(10, 4))
plt.plot(caminhadas_m.T, color='gray', alpha=0.02)
plt.title(f'Evolução de {m} Caminhadas Aleatórias Simultâneas')
plt.xlabel('Número de Passos')
plt.ylabel('Posição')
plt.grid(True, alpha=0.3)
plt.show()

# =====================================================================
# 3. Distribuição Final vs Gaussiana vs Discreta Exata COM BARRA DE ERRO
# =====================================================================
posicoes_finais = caminhadas_m[:, -1]

# Cálculo dos valores teóricos da Gaussiana
media_teorica = n * (2*p - 1)
desvio_padrao_teorico = np.sqrt(4 * n * p * (1 - p))

# Cálculo da Distribuição Discreta Exata (Binomial)
x_discrete = np.arange(-n, n + 1, 2)
n_plus = (n + x_discrete) // 2
probabilidade_exata = binom.pmf(n_plus, n, p)

plt.figure(figsize=(10, 5))

bins_certos = np.arange(-n - 1, n + 3, 2)
pesos = np.ones_like(posicoes_finais) / m

# 1. Salvar os retornos do histograma
hist_vals, bins, patches = plt.hist(
    posicoes_finais,
    bins=bins_certos,
    weights=pesos,
    alpha=0.6,
    color='skyblue',
    edgecolor='black',
    label='Simulação (Empírica)'
)

# 2. Calcular os centros das barras (onde o erro será plotado no eixo X)
bin_centers = 0.5 * (bins[:-1] + bins[1:])

# 3. Calcular o erro padrão para as proporções empíricas
# Como hist_vals já são probabilidades (devido aos weights), a fórmula é direta:
yerr = np.sqrt(hist_vals * (1 - hist_vals) / m)

# 4. Plotar as barras de erro
plt.errorbar(
    bin_centers,
    hist_vals,
    yerr=yerr,
    fmt='none',
    ecolor='black',
    capsize=2,
    alpha=0.8,
    label='Erro Padrão'
)

# Plot da Distribuição Discreta Exata (Binomial)
plt.plot(x_discrete, probabilidade_exata, 'go', markersize=5, label='Distribuição Exata (Binomial)')
plt.vlines(x_discrete, 0, probabilidade_exata, colors='green', linestyles='dotted', alpha=0.5)

# Curva Gaussiana Teórica
eixo_x = np.linspace(media_teorica - 4*desvio_padrao_teorico, media_teorica + 4*desvio_padrao_teorico, 1000)
curva_teorica = norm.pdf(eixo_x, media_teorica, desvio_padrao_teorico) * 2
plt.plot(eixo_x, curva_teorica, 'r-', linewidth=2, label='Aproximação Gaussiana (TLC)')

plt.title(f'Distribuição das Posições Finais (p = {p}, n = {n}, m = {m})')
plt.xlabel('Posição Final')
plt.ylabel('Probabilidade')
plt.xlim(media_teorica - 4*desvio_padrao_teorico, media_teorica + 4*desvio_padrao_teorico)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Visualizando a Evolução Temporal: Deslocamento e Alargamento da Distribuição

Para compreender a dinâmica de um processo difusivo ou de uma caminhada aleatória, é fundamental observar como a distribuição de probabilidade das posições se comporta à medida que o tempo (número de passos) passa. Esse fenômeno é governado por dois conceitos principais:

1. **O Deslocamento (Advecção/Drift):** Se a probabilidade $p \neq 0.5$, existe uma força de tendência. A média da distribuição $\mu(n) = n(2p-1)$ desloca-se de forma **linear** com o número de passos $n$. Se $p > 0.5$, o centro da distribuição viaja para a direita.
2. **O Alargamento (Difusão):** Mesmo que não haja tendência ($p = 0.5$), a distribuição não fica parada; ela se espalha. O desvio padrão cresce com a raiz quadrada do tempo: $\sigma(n) = \sqrt{4np(1-p)} \propto \sqrt{n}$. Como a área total deve ser mantida constante (igual a 1), para a curva ficar mais larga, o seu pico central obrigatoriamente precisa ficar **mais baixo**.

Abaixo, coletamos a posição de 10.000 caminhantes em 4 momentos diferentes ($n = 10, 50, 150, 500$) para ver esses dois efeitos acontecerem simultaneamente.

```python


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# --- Parâmetros ---
m = 10000          # Número de realizações (alto para curvas lisas)
n_max = 500        # Número máximo de passos
p = 0.5           # Probabilidade assimétrica para evidenciar o deslocamento

# Passos de tempo (snapshots) onde registraremos a distribuição
instantes = [10, 50, 150, 500]
cores = ['#2ca02c', '#ff7f0e', '#1f77b4', '#d62728'] # Verde, Laranja, Azul, Vermelho

# =====================================================================
# Simulação Computacional
# =====================================================================
# Matriz de passos aleatórios (m caminhantes x n_max passos)
passos = np.where(np.random.rand(m, n_max) < p, 1, -1)
# Posições ao longo do tempo (incluindo a origem no passo 0)
caminhadas = np.concatenate((np.zeros((m, 1)), np.cumsum(passos, axis=1)), axis=1)

# =====================================================================
# Construção do Gráfico de Evolução
# =====================================================================
plt.figure(figsize=(12, 6))

for t, cor in zip(instantes, cores):
    # Extrai a posição de todos os m caminhantes no instante t
    posicoes_no_instante = caminhadas[:, t]

    # --- 1. Histograma Empírico ---
    bins = np.arange(min(posicoes_no_instante) - 1, max(posicoes_no_instante) + 3, 2)
    pesos = np.ones_like(posicoes_no_instante) / m

    plt.hist(posicoes_no_instante, bins=bins, weights=pesos, histtype='step',
             linewidth=2, color=cor, alpha=0.8, label=f'Simulação no Passo {t}')

    # --- 2. Curva Gaussiana Teórica Correspondente ---
    media_t = t * (2*p - 1)
    sigma_t = np.sqrt(4 * t * p * (1 - p))

    eixo_x = np.linspace(media_t - 4*sigma_t, media_t + 4*sigma_t, 500)
    gaussiana_t = norm.pdf(eixo_x, media_t, sigma_t) * 2

    plt.plot(eixo_x, gaussiana_t, color=cor, linestyle='--', linewidth=1.5)
    plt.fill_between(eixo_x, gaussiana_t, alpha=0.05, color=cor)

# Customização do Gráfico
plt.title(f'Evolução Temporal da Distribuição (p = {p})', fontsize=14, fontweight='bold')
plt.xlabel('Posição na Rede ($x$)', fontsize=12)
plt.ylabel('Probabilidade de Encontrar o Caminhante', fontsize=12)
plt.grid(True, alpha=0.2, linestyle=':')
plt.legend(fontsize=10, loc='upper left')

# CORRIGIDO: Adicionado o prefixo 'r' antes das strings para o Matplotlib tratar o LaTeX isoladamente
plt.annotate(r'Deslocamento da Média (Drift) $\rightarrow$', xy=(120, 0.12), fontsize=11, color='dimgray')
plt.annotate(r'$\leftarrow$ Alargamento (Difusão) e Achatamento $\rightarrow$', xy=(-10, 0.04), fontsize=11, color='dimgray')

plt.ylim(0, 0.30)
plt.xlim(-250, 250)
plt.show()

# Caminhada Aleatória Discreta em 2D vs. Gaussiana Bidimensional

Ao expandirmos a caminhada aleatória para duas dimensões em uma rede quadrada, o caminhante parte da origem $(0,0)$ e possui 4 movimentos possíveis a cada passo, todos com probabilidade $p = 0.25$:
* Direita: $(+1, 0)$
* Esquerda: $(-1, 0)$
* Cima: $(0, +1)$
* Baixo: $(0, -1)$

### A Teoria Discreta e Contínua em 2D
A estatística de cada eixo pode ser aproximada de forma independente. Para uma caminhada simétrica de $n$ passos:
* **Média em ambos os eixos:** $\mu_x = \mu_y = 0$
* **Variância por eixo:** Como o caminhante só se move em *um* dos eixos a cada passo, a variância acumulada é distribuída igualmente, resultando em $\sigma^2 = n/2$ para cada coordenada. Portanto, o desvio padrão de cada eixo será $\sigma = \sqrt{n/2}$.

Como o espaço amostral discreto bidimensional possui restrições de paridade em ambos os eixos ($\Delta x = 2$ e $\Delta y = 2$), a área elementar de cada ponto na grade discreta é $\Delta A = 2 \times 2 = 4$. Para que a Função Densidade de Probabilidade (PDF) contínua da Gaussiana 2D corresponda à massa de probabilidade discreta, devemos multiplicar o resultado contínuo por **4**.

Abaixo, simulamos a posição final de 20.000 caminhantes e comparamos o histograma 2D empírico com a superfície matemática teórica da Gaussiana.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Parâmetros da Simulação ---
n_passos = 1000     # Número de passos (N)
n_caminhantes = 5000 # Quantidade de caminhantes para a estatística
ell = 1.0           # Espaçamento da rede (l)

# =====================================================================
# 1. Geração das Caminhadas Aleatórias na Rede Quadrada
# =====================================================================
# As 4 direções possíveis: Direita, Esquerda, Cima, Baixo
direcoes_possiveis = np.array([[ell, 0], [-ell, 0], [0, ell], [0, -ell]])

# Sorteia os índices (0, 1, 2 ou 3) de forma equiprovável (p = 1/4)
# Matriz gerada terá o formato: (n_passos, n_caminhantes)
indices_sorteados = np.random.choice([0, 1, 2, 3], size=(n_passos, n_caminhantes))

# Mapeia os índices sorteados para os vetores de passos reais (dx, dy)
passos = direcoes_possiveis[indices_sorteados]
dx = passos[:, :, 0]
dy = passos[:, :, 1]

# Insere a origem (0,0) no início de todas as trajetórias e faz a soma acumulada
x = np.vstack((np.zeros(n_caminhantes), np.cumsum(dx, axis=0)))
y = np.vstack((np.zeros(n_caminhantes), np.cumsum(dy, axis=0)))

# =====================================================================
# 2. Análise Estatística (Deslocamento Quadrático Médio)
# =====================================================================
# Calcula R^2 para cada caminhante em cada passo: R^2 = X^2 + Y^2
r2_individual = x**2 + y**2

# Faz a média aritmética sobre todos os caminhantes (eixo das colunas)
r2_medio_simulado = np.mean(r2_individual, axis=1)

# Curva Teórica deduzida no texto: <R_N^2> = N * l^2
passos_eixo = np.arange(0, n_passos + 1)
r2_teorico = passos_eixo * (ell**2)

# =====================================================================
# 3. Construção dos Gráficos Ilustrativos
# =====================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# --- Gráfico 1: Trajetórias Discretas no Espaço ---
n_visualizar = 4  # Vamos plotar apenas as primeiras trajetórias para não poluir
for i in range(n_visualizar):
    ax1.plot(x[:, i], y[:, i], linewidth=1.2, drawstyle='default', label=f'Caminhante {i+1}')
    ax1.plot(x[-1, i], y[-1, i], 'ko', markersize=4) # Ponto final

ax1.plot(0, 0, 'ro', markersize=8, label='Origem (0,0)') # Ponto inicial
ax1.set_title('Trajetórias Típicas na Rede Quadrada 2D\n(Movimentos Discretos a Primeiros-Vizinhos)')
ax1.set_xlabel('Posição $X$')
ax1.set_ylabel('Posição $Y$')
ax1.grid(True, alpha=0.3)
ax1.legend()
ax1.axis('equal') # Garante que a proporção dos eixos X e Y seja idêntica

# --- Gráfico 2: Validação do Deslocamento Quadrático Médio ---
ax2.plot(passos_eixo, r2_medio_simulado, 'g-', linewidth=3, label=f'Simulação ({n_caminhantes} amostras)')
ax2.plot(passos_eixo, r2_teorico, 'r--', linewidth=1.5, label='Teoria ($\langle R_N^2 \\rangle = N \ell^2$)')
ax2.set_title('Deslocamento Quadrático Médio vs. Número de Passos')
ax2.set_xlabel('Número de Passos ($N$)')
ax2.set_ylabel('$\langle R_N^2 \\rangle$')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal

# --- Parâmetros ---
n = 100        # Número de passos (tempo de caminhada)
m = 1000000     # Número de realizações (caminhantes independentes)

# =====================================================================
# 1. Simulação Computacional da Caminhada 2D
# =====================================================================
# Sorteia uma direção de 0 a 3 para cada passo de cada caminhante
direcoes = np.random.randint(0, 4, size=(m, n))

# Dicionário de movimentos mapeado para matrizes
# 0: Direita (+1,0) | 1: Esquerda (-1,0) | 2: Cima (0,+1) | 3: Baixo (0,-1)
dx = np.where(direcoes == 0, 1, np.where(direcoes == 1, -1, 0))
dy = np.where(direcoes == 2, 1, np.where(direcoes == 3, -1, 0))

# Posição final (soma de todos os passos para cada caminhante)
x_final = np.sum(dx, axis=1)
y_final = np.sum(dy, axis=1)

# =====================================================================
# 2. Cálculo da Gaussiana Bidimensional Teórica
# =====================================================================
# Variância em cada eixo é n/2 para a caminhada simétrica na rede
sigma_eixo = np.sqrt(n / 2)

# Cria a malha (grid) para plotar a superfície teórica
limite = int(3 * sigma_eixo)
x_grade = np.arange(-limite, limite + 1, 2) # Passos de 2 em 2 devido à paridade
y_grade = np.arange(-limite, limite + 1, 2)
X, Y = np.meshgrid(x_grade, y_grade)

# Define a Gaussiana 2D usando Scipy
pos = np.dstack((X, Y))
mu = [0, 0]
cov = [[sigma_eixo**2, 0], [0, sigma_eixo**2]] # Matriz de covariância diagonal (eixos independentes)
gaussiana_2d = multivariate_normal(mu, cov)

# Multiplicamos por 4 (dx * dy = 2 * 2) para ajustar a densidade à massa discreta
Z_teorico = gaussiana_2d.pdf(pos) * 4

# =====================================================================
# 3. Construção do Gráfico 3D de Comparação
# =====================================================================
# =====================================================================
# 3. Construção do Gráfico 3D de Comparação (VERSÃO CORRIGIDA)
# =====================================================================
fig = plt.figure(figsize=(14, 7))

# --- Subplot 1: Histograma Empírico (Dados da Simulação) ---
ax1 = fig.add_subplot(1, 2, 1, projection='3d')

bins_x = np.arange(-limite - 1, limite + 3, 2)
bins_y = np.arange(-limite - 1, limite + 3, 2)

hist, xedges, yedges = np.histogram2d(x_final, y_final, bins=[bins_x, bins_y])
hist_prob = hist / m  # Fração de caminhantes em cada ponto

# CORREÇÃO CRUCIAL: Gerar o meshgrid sem 'ij' e transpor o histograma antes do ravel
xpos, ypos = np.meshgrid(xedges[:-1] + 1, yedges[:-1] + 1)
xpos = xpos.ravel()
ypos = ypos.ravel()
zpos = np.zeros_like(xpos)

dx_barras = dy_barras = 1.5 * np.ones_like(xpos)
# .T garante que a contagem do eixo X case com o achatamento padrão do meshgrid
dz_barras = hist_prob.T.ravel()

# Renderiza as barras localizadas corretamente
ax1.bar3d(xpos, ypos, zpos, dx_barras, dy_barras, dz_barras, color='skyblue', alpha=0.7, edgecolor='black')
ax1.set_title(f'Distribuição Empírica\n(Simulação com {m} caminhantes)')
ax1.set_xlabel('Eixo X')
ax1.set_ylabel('Eixo Y')
ax1.set_zlabel('Probabilidade')
ax1.set_zlim(0, np.max(Z_teorico)*1.2)
ax1.view_init(elev=30, azim=-45) # Rotaciona para melhor visualização matemática

# --- Subplot 2: Superfície Contínua Teórica (Gaussiana 2D) ---
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
superficie = ax2.plot_surface(X, Y, Z_teorico, cmap='inferno', edgecolor='none', alpha=0.8)
ax2.set_title(f'Aproximação Gaussiana Teórica\n(TLC Corrigido para Área $\Delta A = 4$)')
ax2.set_xlabel('Eixo X')
ax2.set_ylabel('Eixo Y')
ax2.set_zlabel('Probabilidade')
ax2.set_zlim(0, np.max(Z_teorico)*1.2)
ax2.view_init(elev=30, azim=-45) # Mesma orientação do primeiro gráfico

fig.colorbar(superficie, ax=ax2, shrink=0.5, aspect=10, label='Massa de Probabilidade')
plt.suptitle(f'Caminhada Aleatória 2D após {n} passos', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Parâmetros Fixos do Sistema ---
L = 30              # Tamanho da rede quadrada (L x L)
N_passos_MCT = 50   # Passos de Monte Carlo por partícula (tempo da difusão)
densidades = np.array([0.05, 0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75, 0.85])

# Vetores de movimentos para os 4 primeiros-vizinhos
direcoes = np.array([[1, 0], [-1, 0], [0, 1], [0, -1]])

# Listas para armazenar os resultados estatísticos
D_experimentais = []
D_erros = []  # Lista que guardará a barra de erro (erro padrão) para cada densidade

print("Iniciando a simulação do Gás na Rede com cálculo de incertezas...")

for rho in densidades:
    # 1. INICIALIZAÇÃO
    M = int(rho * L**2) # Número de partículas para a densidade atual

    # Criar a rede vazia (0) e sortear posições únicas para as partículas (1)
    rede = np.zeros((L, L), dtype=int)
    indices_sitos = np.random.choice(L**2, size=M, replace=False)

    pos_x = indices_sitos % L
    pos_y = indices_sitos // L

    for i in range(M):
        rede[pos_x[i], pos_y[i]] = 1

    # Posições reais acumuladas para o cálculo do deslocamento líquido
    x_real = pos_x.astype(float)
    y_real = pos_y.astype(float)

    x_inicial = np.copy(x_real)
    y_inicial = np.copy(y_real)

    N_total_tentativas = N_passos_MCT * M

    # 2. EVOLUÇÃO TEMPORAL (ALGORITMO DE MONTE CARLO)
    for _ in range(N_total_tentativas):
        p_id = np.random.randint(0, M)
        dir_id = np.random.randint(0, 4)
        dx, dy = direcoes[dir_id]

        nova_pos_x = int((pos_x[p_id] + dx) % L)
        nova_pos_y = int((pos_y[p_id] + dy) % L)

        if rede[nova_pos_x, nova_pos_y] == 0:
            rede[pos_x[p_id], pos_y[p_id]] = 0
            rede[nova_pos_x, nova_pos_y] = 1

            pos_x[p_id] = nova_pos_x
            pos_y[p_id] = nova_pos_y

            x_real[p_id] += dx
            y_real[p_id] += dy

    # 3. AMOSTRAGEM E CÁLCULO DAS BARRAS DE ERRO
    # Deslocamento quadrático de CADA partícula individualmente
    r2_individual = (x_real - x_inicial)**2 + (y_real - y_inicial)**2

    # Média amostral de R^2
    r2_medio = np.mean(r2_individual)

    # Desvio padrão amostral dos deslocamentos das partículas
    r2_std = np.std(r2_individual, ddof=1) if M > 1 else 0

    # Erro padrão da média (Incerteza associada ao tamanho da amostra M)
    r2_erro_padrao = r2_std / np.sqrt(M)

    # Coeficiente de Difusão: D = <R^2> / (4 * N_passos)
    D_efetivo = r2_medio / (4 * N_passos_MCT)

    # Propagação do erro linear para D
    D_erro_efetivo = r2_erro_padrao / (4 * N_passos_MCT)

    D_experimentais.append(D_efetivo)
    D_erros.append(D_erro_efetivo)

# =====================================================================
# 4. PLOTAGEM DO GRÁFICO COM BARRAS DE ERRO
# =====================================================================
D0 = 0.25
rho_teorico = np.linspace(0, 1, 100)
D_campo_medio = D0 * (1 - rho_teorico)

plt.figure(figsize=(9, 6))

# Linha teórica de campo médio
plt.plot(rho_teorico, D_campo_medio, 'r--', linewidth=2, label=r'Teoria de Campo Médio: $D_0(1-\rho)$')

# Pontos experimentais usando 'errorbar' para incluir as barras de erro verticais (yerr)
plt.errorbar(densidades, D_experimentais, yerr=D_erros, fmt='bo-',
             ecolor='black', elinewidth=1.5, capsize=4, linewidth=2,
             label='Simulação (Gás na Rede) com Erro Padrão')

plt.title('Efeito de Exclusão Volumétrica: Coeficiente de Difusão vs Densidade', fontsize=12)
plt.xlabel(r'Densidade de Partículas ($\rho$)', fontsize=11)
plt.ylabel(r'Coeficiente de Difusão Efetivo ($D$)', fontsize=11)
plt.xlim(0, 1)
plt.ylim(0, 0.3)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)

plt.text(0.4, 0.05, 'As barras de erro pretas representam\no Erro Padrão da Média calculado\na partir da população de partículas.',
         color='black', bbox=dict(facecolor='lightgray', alpha=0.5), fontsize=10)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# --- Parâmetros ---
n = 100       # Número de passos
m = 100000    # Número de caminhantes
p = 0.3       # Probabilidade de ir para a direita
ell = 1.5     # Comprimento médio do passo exponencial

# =====================================================================
# Simulação Computacional (Espaço Contínuo)
# =====================================================================
# 1. Sorteia as direções (+1 ou -1)
direcoes = np.where(np.random.rand(m, n) < p, 1, -1)

# 2. Sorteia os comprimentos com base na distribuição exponencial
comprimentos = np.random.exponential(scale=ell, size=(m, n))

# 3. O passo real é a multiplicação da direção pelo comprimento
passos_continous = direcoes * comprimentos
posicoes_finais = np.sum(passos_continous, axis=1)

# =====================================================================
# Cálculos Teóricos da Gaussiana
# =====================================================================
media_passo = (2 * p - 1) * ell
variancia_passo = (ell**2) * (4 * p * (1 - p) + 1)

media_teorica = n * media_passo
desvio_padrao_teorico = np.sqrt(n * variancia_passo)

# =====================================================================
# Construção do Gráfico
# =====================================================================
plt.figure(figsize=(10, 5))

# 1. Plotar o histograma e salvar as informações geradas
hist_vals, bins, _ = plt.hist(
    posicoes_finais, bins=50, density=True, alpha=0.6,
    color='lightgreen', edgecolor='black', label='Simulação (Espaço Contínuo)'
)

# 2. Obter os centros e as larguras dos bins
bin_centers = 0.5 * (bins[:-1] + bins[1:])
bin_widths = bins[1:] - bins[:-1]

# 3. Calcular a probabilidade real associada a cada bin e propagar o erro para a densidade
prob_bins = hist_vals * bin_widths
yerr = np.sqrt(prob_bins * (1 - prob_bins) / m) / bin_widths

# 4. Adicionar a barra de erro
plt.errorbar(
    bin_centers,
    hist_vals,
    yerr=yerr,
    fmt='none',
    ecolor='black',
    capsize=2,
    alpha=0.7,
    label='Erro Padrão'
)

# Curva Gaussiana Teórica PURA
eixo_x = np.linspace(media_teorica - 4*desvio_padrao_teorico, media_teorica + 4*desvio_padrao_teorico, 1000)
pdf_teorica = norm.pdf(eixo_x, media_teorica, desvio_padrao_teorico)

plt.plot(eixo_x, pdf_teorica, 'r-', linewidth=2.5, label='Aproximação Gaussiana (TLC Puro)')

plt.title(f'Caminhada Aleatória com Passos Exponenciais Contínuos (n={n})')
plt.xlabel('Posição Final ($x$)')
plt.ylabel('Densidade de Probabilidade')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Parâmetros ---
n = 1000      # Número de passos para cada caminhante
m = 1         # Número de caminhantes simultâneos para visualizar
ell = 1.0     # Comprimento fixo de cada passo

# 1. Sorteia os ângulos uniformemente entre 0 e 2*pi para a matriz (passos, caminhantes)
thetas = np.random.uniform(0, 2 * np.pi, size=(n, m))

# 2. Calcula as projeções cartesianas de cada passo individual
dx = ell * np.cos(thetas)
dy = ell * np.sin(thetas)

# 3. Garante que todos comecem na origem (0,0) e faz a soma acumulada das posições
x = np.vstack((np.zeros(m), np.cumsum(dx, axis=0)))
y = np.vstack((np.zeros(m), np.cumsum(dy, axis=0)))

# =====================================================================
# Construção do Gráfico Bidimensional
# =====================================================================
plt.figure(figsize=(8, 8))

# Plota a trajetória colorida de cada caminhante
for i in range(m):
    plt.plot(x[:, i], y[:, i], linewidth=1.5, label=f'Caminhante {i+1}')
    # Marcador discreto na posição final de cada um
    plt.plot(x[-1, i], y[-1, i], 'ko', markersize=5)

# Destaca a origem comum (0,0) de onde todos partiram
plt.plot(0, 0, 'ro', markersize=8, label='Origem (0,0)')

plt.title(f'Caminhada Aleatória no Contínuo 2D (Problema de Pearson)\n$N = {n}$ passos com comprimento $\ell = {ell}$')
plt.xlabel('Posição $X$')
plt.ylabel('Posição $Y$')

# Linhas de referência para os eixos centrais
plt.axhline(0, color='black', linestyle=':', alpha=0.3)
plt.axvline(0, color='black', linestyle=':', alpha=0.3)

plt.grid(True, alpha=0.3)
plt.legend()

# CRUCIAL: Mantém a proporção 1:1 dos eixos para não distorcer o espaço físico
plt.axis('equal')

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import levy_stable, norm

# --- Parâmetros ---
n = 50        # Número de passos
m = 10000     # Número de caminhantes
alpha = 1.3   # Expoente de Lévy (deve ser entre 0 e 2. Se for 2, vira Gaussiana)

# =====================================================================
# Simulação Computacional do Voo de Lévy
# =====================================================================
# 1. Sorteia direções simétricas
direcoes = np.random.choice([-1, 1], size=(m, n))

# 2. Sorteia comprimentos usando a distribuição de Pareto (Lei de Potência)
u = np.random.rand(m, n)
comprimentos = (u) ** (-1.0 / alpha)

passos_levy = direcoes * comprimentos
posicoes_finais = np.sum(passos_levy, axis=1)

# =====================================================================
# Visualização Visual da Trajetória (Efeito de Super-Saltos)
# =====================================================================
passos_exemplo = np.random.choice([-1, 1], size=500) * (np.random.rand(500) ** (-1.0 / alpha))
trajetoria_exemplo = np.cumsum(passos_exemplo)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(trajetoria_exemplo, color='purple', linewidth=1)
ax1.title.set_text("Trajetória Típica de um Voo de Lévy (Nota os Super-Saltos)")
ax1.set_xlabel("Tempo (Passos)")
ax1.set_ylabel("Posição")
ax1.grid(True, alpha=0.3)

# =====================================================================
# Histograma das Posições Finais vs Teoria Estável (COM BARRA DE ERRO)
# =====================================================================
limite_visual = 200
dados_filtrados = posicoes_finais[np.abs(posicoes_finais) < limite_visual]

# 1. Capturar o retorno do histograma
hist_vals, bins, _ = ax2.hist(
    dados_filtrados, bins=100, density=True, alpha=0.6,
    color='plum', edgecolor='purple', label='Simulação de Lévy'
)

# 2. Calcular larguras, centros e o tamanho da amostra efetiva (filtrada)
bin_centers = 0.5 * (bins[:-1] + bins[1:])
bin_widths = bins[1:] - bins[:-1]
m_filtrado = len(dados_filtrados)  # Usamos a contagem dos dados filtrados

# 3. Converter a densidade para probabilidade por bin e calcular o erro padrão
prob_bins = hist_vals * bin_widths
yerr = np.sqrt(prob_bins * (1 - prob_bins) / m_filtrado) / bin_widths

# 4. Adicionar a barra de erro ao eixo 'ax2'
ax2.errorbar(
    bin_centers, hist_vals, yerr=yerr, fmt='none',
    ecolor='purple', capsize=1.5, alpha=0.6, label='Erro Padrão'
)

# Plot da Distribuição Alfa-Estável Teórica correspondente
eixo_x = np.linspace(-limite_visual, limite_visual, 2000)
pdf_estavel = levy_stable.pdf(eixo_x, alpha, 0, scale=n**(1/alpha))
ax2.plot(eixo_x, pdf_estavel, 'r-', linewidth=2.5, label=f'Teoria Alfa-Estável ($\\alpha$={alpha})')

# Plot de uma Gaussiana ingênua
media_amostral = np.mean(posicoes_finais)
std_amostral = np.std(dados_filtrados)
ax2.plot(eixo_x, norm.pdf(eixo_x, media_amostral, std_amostral), 'b--', label='Gaussiana Equivalente (Falha)')

ax2.title.set_text("Distribuição Final: Cauda Pesada vs Gaussiana")
ax2.set_xlabel("Posição Final ($x$)")
ax2.set_ylabel("Densidade de Probabilidade")
ax2.set_xlim(-limite_visual, limite_visual)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Visualização de Trajetórias 2D: Passos Exponenciais vs. Voos de Lévy

Para compreender a diferença fundamental entre processos difusivos normais e anômalos no espaço contínuo bidimensional, podemos analisar a geometria de suas trajetórias individuais ao longo do tempo (número de passos $n$).

* **Caminhada com Passos Exponenciais:** O comprimento do passo $L$ segue uma distribuição exponencial de média $\ell$. Como a variância é finita, os passos possuem uma escala de tamanho característica. O caminhante explora o espaço de maneira uniforme e contígua.
* **Voo de Lévy:** O comprimento do passo $L$ segue uma distribuição de Pareto (lei de potência) com expoente $\alpha < 2$. A variância infinita introduz eventos raros de magnitudes extremas (super-saltos). O espaço é explorado através de estruturas auto-similares em forma de ilhas de busca local interconectadas por saltos de longa distância.

Em ambos os casos, a direção do movimento a cada passo é completamente isotrópica, ou seja, sorteada uniformemente entre $0$ e $2\pi$ radianos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Parâmetros globais ---
n_passos = 1000      # Número de passos para desenhar a trajetória
ell = 1.0            # Comprimento médio do passo para a exponencial
alpha = 1.2          # Expoente de Lévy (1 < alpha < 2 para super-saltos nítidos)

# Configuração da semente para reprodutibilidade
np.random.seed(42)

# =====================================================================
# 1. Geração da Trajetória com Passos Exponenciais
# =====================================================================
# Sorteio de ângulos uniformes entre 0 e 2*pi
angulos_exp = np.random.uniform(0, 2 * np.pi, n_passos)
# Sorteio de comprimentos seguindo a distribuição exponencial
comprimentos_exp = np.random.exponential(scale=ell, size=n_passos)

# Decomposição nos eixos X e Y
dx_exp = comprimentos_exp * np.cos(angulos_exp)
dy_exp = comprimentos_exp * np.sin(angulos_exp)

# Posições cumulativas ao longo do tempo (começando em 0,0)
x_exp = np.concatenate(([0], np.cumsum(dx_exp)))
y_exp = np.concatenate(([0], np.cumsum(dy_exp)))

# =====================================================================
# 2. Geração da Trajetória com Voo de Lévy
# =====================================================================
angulos_levy = np.random.uniform(0, 2 * np.pi, n_passos)
# Sorteio de comprimentos seguindo a Lei de Potência (Pareto) via inversão da CDF
u = np.random.uniform(0.01, 1.0, n_passos) # evita divisão por zero extrema
comprimentos_levy = (u) ** (-1.0 / alpha)

dx_levy = comprimentos_levy * np.cos(angulos_levy)
dy_levy = comprimentos_levy * np.sin(angulos_levy)

x_levy = np.concatenate(([0], np.cumsum(dx_levy)))
y_levy = np.concatenate(([0], np.cumsum(dy_levy)))

# =====================================================================
# 3. Construção dos Gráficos Comparativos
# =====================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# --- Gráfico 1: Passos Exponenciais ---
ax1.plot(x_exp, y_exp, color='#1f77b4', linewidth=1.2, alpha=0.8, label='Trajetória')
ax1.scatter(x_exp[0], y_exp[0], color='green', s=100, zorder=5, label='Início (0,0)')
ax1.scatter(x_exp[-1], y_exp[-1], color='red', s=100, zorder=5, label='Fim')

ax1.set_title("Caminhada Aleatória 2D (Passos Exponenciais)", fontsize=13, fontweight='bold')
ax1.set_xlabel("Posição X")
ax1.set_ylabel("Posição Y")
ax1.grid(True, alpha=0.2, linestyle=':')
ax1.legend(loc='upper left')
ax1.axis('equal') # Mantém a proporção de escala igual para não distorcer o passo

# --- Gráfico 2: Voo de Lévy ---
ax2.plot(x_levy, y_levy, color='#9467bd', linewidth=1.2, alpha=0.8, label='Trajetória')
ax2.scatter(x_levy[0], y_levy[0], color='green', s=100, zorder=5, label='Início (0,0)')
ax2.scatter(x_levy[-1], y_levy[-1], color='red', s=100, zorder=5, label='Fim')

ax2.set_title(f"Voo de Lévy 2D (Lei de Potência, $\\alpha$={alpha})", fontsize=13, fontweight='bold')
ax2.set_xlabel("Posição X")
ax2.set_ylabel("Posição Y")
ax2.grid(True, alpha=0.2, linestyle=':')
ax2.legend(loc='upper left')
ax2.axis('equal')

plt.suptitle(f"Comparativo Espacial de Caminhadas Contínuas em 2D ({n_passos} passos)", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Parâmetros globais ---
n_passos = 1000      # Número de passos para desenhar a trajetória
ell = 1.0            # Comprimento médio do passo para a exponencial
alpha = 1.2          # Expoente de Lévy (1 < alpha < 2)

# Configuração da semente para reprodutibilidade
np.random.seed(42)

# =====================================================================
# 1. Geração da Trajetória com Passos Exponenciais (Normal)
# =====================================================================
angulos_exp = np.random.uniform(0, 2 * np.pi, n_passos)
comprimentos_exp = np.random.exponential(scale=ell, size=n_passos)

dx_exp = comprimentos_exp * np.cos(angulos_exp)
dy_exp = comprimentos_exp * np.sin(angulos_exp)

x_exp = np.concatenate(([0], np.cumsum(dx_exp)))
y_exp = np.concatenate(([0], np.cumsum(dy_exp)))

# =====================================================================
# 2. Geração da Trajetória com Voo de Lévy (Anômala)
# =====================================================================
angulos_levy = np.random.uniform(0, 2 * np.pi, n_passos)
u = np.random.uniform(0.01, 1.0, n_passos)
comprimentos_levy = (u) ** (-1.0 / alpha)

dx_levy = comprimentos_levy * np.cos(angulos_levy)
dy_levy = comprimentos_levy * np.sin(angulos_levy)

x_levy = np.concatenate(([0], np.cumsum(dx_levy)))
y_levy = np.concatenate(([0], np.cumsum(dy_levy)))

# =====================================================================
# 3. Construção do Gráfico Sobreposto (Versão Segura e Limpa)
# =====================================================================
plt.figure(figsize=(10, 8))

# Plotar as linhas das trajetórias
plt.plot(x_exp, y_exp, color='#1f77b4', linewidth=1.2, alpha=0.8,
         label='Passos Exponenciais (Difusão Normal - Variância Finita)')

plt.plot(x_levy, y_levy, color='#9467bd', linewidth=1.2, alpha=0.8,
         label=fr'Voo de Lévy (Difusão Anômala - $\alpha$={alpha} - Variância Infinita)')

# Marcadores de início e fim das caminhadas
plt.scatter(x_levy[0], y_levy[0], color='green', s=120, zorder=5, label='Início Comum (0,0)')
plt.scatter(x_exp[-1], y_exp[-1], color='#1f77b4', s=100, zorder=4, edgecolor='black', label='Fim Exponencial')
plt.scatter(x_levy[-1], y_levy[-1], color='#9467bd', s=100, zorder=4, edgecolor='black', label='Fim Lévy')

# Customização dos eixos e títulos (sem formatações complexas que quebram string)
plt.title("SOBREPOSICAO DE TRAJETORIAS: EXPONENCIAL vs. VOO DE LEVY", fontsize=14, fontweight='bold')
plt.xlabel("Posicao X", fontsize=12)
plt.ylabel("Posicao Y", fontsize=12)
plt.grid(True, alpha=0.2, linestyle=':')
plt.legend(loc='upper left', fontsize=10)
plt.axis('equal')

plt.tight_layout()
plt.show()